In [0]:
schemalocation = "/Volumes/ev_spark/myvol/checkpoint/schema/ontario_ev"
sourcepath = "/Volumes/ev_spark/myvol/landing/ontario_ev_pop/"
chkptpath = "/Volumes/ev_spark/myvol/checkpoint/chkpt/ontario_ev/"

In [0]:
def readOnEvToDf():
  print("Reading ON EV to DF")
  from pyspark.sql.functions import current_timestamp
  from pyspark.sql.types import StructType, StructField, StringType
  schema = StructType([StructField("_id", StringType(), True),
                       StructField("FSA", StringType(), True),
                       StructField("BEV", StringType(), True),
                       StructField("PHEV", StringType(), True),
                       StructField("Total_EV", StringType(), True)])
  readRawONEV_df = (spark.readStream
                      .format("cloudFiles")
                      .option("cloudFiles.format", "csv")
                      .option("cloudFiles.schemaLocation", schemalocation)
                      .option("header", "true")
                      .schema(schema)
                      .load(sourcepath)
                      .withColumn("extraction_date", current_timestamp())
  )
  print("Read successful")
  print("************************")
  return readRawONEV_df

In [0]:
readRawONEV_df.printSchema()

In [0]:
def writeOnEvToDelta(df):
  print("Starting to write ON EV To Delta Table")
  (df.writeStream
  .format("delta")
  .option("checkpointLocation", chkptpath)
  .outputMode("append")
  .trigger(availableNow= True)
  .toTable("ev_spark.bronze.ontario_ev")
  )
  print("Write successful")
  print("************************")

In [0]:
readRawOnEv_df = readOnEvToDf()
writeOnEvToDelta(readRawOnEv_df)

In [0]:
%sql
select * from ev_spark.bronze.ontario_ev